# Stage 3, Part B: Load the Parquet files into Snowflake
Uploads the cleaned Parquet files from Stage 2 to a Snowflake stage, then loads them into one raw table, `RAW.REVIEWS`.

**Only one teammate runs this, once.** Everyone else reads the table from Snowflake.

**Before you start**
- Part A (role, schemas, permissions) must be done, otherwise `PROJECT_ROLE` will not exist.
- Stage 2 must be finished (`data/processed/reviews.parquet`).
- The upload is several GB, so expect roughly 10-30 minutes depending on your internet speed. Do not close the notebook while it runs.
- Set `LOAD_FULL = False` in the settings cell for a quick test with the 10% sample.

In [ ]:
%pip install -q snowflake-connector-python

## 1. Settings

In [ ]:
import getpass
from pathlib import Path
import snowflake.connector

ACCOUNT   = "SDTPFIN-KG72944"
USER      = "xiezuoye33"          # your Snowflake user name (SELECT CURRENT_USER() shows it)
ROLE      = "PROJECT_ROLE"
WAREHOUSE = "COMPUTE_WH"
DATABASE  = "DATA226GROUPPROJECT"
SCHEMA    = "RAW"

LOAD_FULL = True    # False = load only the 10% sample (faster, good for a first test)

if LOAD_FULL:
    LOCAL_DIR, TABLE, PREFIX = Path("data/processed/reviews.parquet"), "REVIEWS", "full"
else:
    LOCAL_DIR, TABLE, PREFIX = Path("data/processed/reviews_sample.parquet"), "REVIEWS_SAMPLE", "sample"

CATEGORIES = ["Books", "Music", "Video_DVD", "Digital_Ebook"]

assert LOCAL_DIR.exists(), f"{LOCAL_DIR} not found. Run the Stage 2 notebook first."
total_gb = 0
for cat in CATEGORIES:
    files = list((LOCAL_DIR / f"category={cat}").glob("*.parquet"))
    size = sum(f.stat().st_size for f in files) / 1e9
    total_gb += size
    print(f"{cat}: {len(files)} files, {size:.2f} GB")
    assert files, f"No Parquet files found for {cat}"
print(f"Total to upload: {total_gb:.2f} GB")

## 2. Connect to Snowflake
Type your password (or token) when asked. It is not saved in the notebook.

In [ ]:
conn = snowflake.connector.connect(
    account=ACCOUNT, user=USER,
    password=getpass.getpass("Snowflake password or token: "),
    role=ROLE, warehouse=WAREHOUSE, database=DATABASE, schema=SCHEMA,
)
cur = conn.cursor()

def run(sql):
    cur.execute(sql)
    return cur.fetchall()

print(run("SELECT CURRENT_USER(), CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_DATABASE(), CURRENT_SCHEMA()")[0])

## 3. Create the stage and the raw table
The stage is a Snowflake folder that holds the uploaded files. Re-running this cell empties the table and starts over.

In [ ]:
run("CREATE STAGE IF NOT EXISTS REVIEWS_STAGE FILE_FORMAT = (TYPE = PARQUET)")

run(f"""
CREATE OR REPLACE TABLE {TABLE} (
    review_id          STRING,
    product_id         STRING,
    customer_id        NUMBER,
    star_rating        NUMBER(1,0),
    helpful_votes      NUMBER,
    total_votes        NUMBER,
    verified_purchase  BOOLEAN,
    review_date        DATE,
    review_year        NUMBER(4,0),
    review_text        STRING,
    word_count         NUMBER,
    category           STRING
)
""")
print(f"Table {SCHEMA}.{TABLE} is ready.")

## 4. Upload the files (the slow step)
Spark stores `category` in the folder name, not inside the files, so we upload one category at a time and fill the column in step 5.

In [ ]:
for cat in CATEGORIES:
    folder = (LOCAL_DIR / f"category={cat}").resolve().as_posix()
    print(f"Uploading {cat} ...")
    res = run(f"PUT 'file://{folder}/*.parquet' @REVIEWS_STAGE/{PREFIX}/{cat}/ "
              f"AUTO_COMPRESS=FALSE PARALLEL=8 OVERWRITE=TRUE")
    status = {}
    for r in res:
        status[r[6]] = status.get(r[6], 0) + 1      # column 6 = status (UPLOADED / SKIPPED)
    print("   ", status)

## 5. Load into the table
`COPY INTO` matches Parquet columns to table columns by name. Then we fill in the `category` column for the rows just loaded.

In [ ]:
for cat in CATEGORIES:
    res = run(f"COPY INTO {TABLE} FROM @REVIEWS_STAGE/{PREFIX}/{cat}/ "
              f"FILE_FORMAT = (TYPE = PARQUET) MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE "
              f"ON_ERROR = ABORT_STATEMENT")
    loaded = sum(r[3] for r in res if len(r) > 3 and isinstance(r[3], int))   # column 3 = rows_loaded
    run(f"UPDATE {TABLE} SET category = '{cat}' WHERE category IS NULL")
    print(f"{cat}: {loaded:,} rows loaded")

## 6. Check the result against Spark

In [ ]:
expected = {"Books": 3105433, "Music": 4748339, "Video_DVD": 5063668, "Digital_Ebook": 5101681}

counts = dict(run(f"SELECT category, COUNT(*) FROM {TABLE} GROUP BY category"))
print(f"{'Category':<15}{'Snowflake':>14}{'Spark (Stage 2)':>18}  Match")
for cat in CATEGORIES:
    n = counts.get(cat, 0)
    ok = "yes" if (LOAD_FULL and n == expected[cat]) else ("-" if not LOAD_FULL else "NO")
    print(f"{cat:<15}{n:>14,}{expected[cat]:>18,}  {ok}")
print(f"{'TOTAL':<15}{sum(counts.values()):>14,}")

bad = run(f"SELECT COUNT(*) FROM {TABLE} WHERE category IS NULL OR review_date IS NULL OR star_rating IS NULL")[0][0]
print("\nRows with a null category, date, or rating (should be 0):", bad)
print("Date range:", run(f"SELECT MIN(review_date), MAX(review_date) FROM {TABLE}")[0])

print("\nRating distribution:")
for rating, n in run(f"SELECT star_rating, COUNT(*) FROM {TABLE} GROUP BY star_rating ORDER BY star_rating"):
    print(f"  {rating} stars: {n:,}")

print("\nSample rows:")
for r in run(f"SELECT review_id, category, star_rating, review_date, LEFT(review_text, 60) FROM {TABLE} LIMIT 3"):
    print(" ", r)

In [ ]:
conn.close()
print("Connection closed.")

**Done with Part B.** The table `DATA226GROUPPROJECT.RAW.REVIEWS` now holds the full cleaned dataset. Part C builds the dbt project on top of it.